# LOB grid sizing: $d^\ast$ vs. $\delta$

For each tick spacing $\delta$, plot the minimum number of levels per side $d^\ast$ required so the grid half-width $d^\ast \delta$ covers the empirical 95 / 98 percentile of $|\Delta\mathrm{mid}|$ between consecutive 5-second snapshots.

The chosen configuration $(d=5,\,\delta=2\,\mathrm{USDT})$ sits on the 98 % curve: it is the minimum-level grid attaining 98 % coverage. Finer $\delta$ proportionally inflates $d^\ast$; coarser $\delta$ collapses to too few levels for near-touch resolution.

In [ ]:
import os, sys
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find repo root
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'lobgrad' / '__init__.py').is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError('Could not find lobgrad package')

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lobgrad.data.loader import load_day_signed

DATA_DIR     = REPO_ROOT / 'data' / 'raw'
SYMBOL       = 'BTCUSDT'
TIME_DIFF_MS = 5_000
DATES        = ['2023-10-01', '2023-10-05', '2023-10-10']   # representative training days

all_anchors = []
for date_str in DATES:
    dt    = datetime.strptime(date_str, '%Y-%m-%d')
    fname = f'{SYMBOL}_{dt.month:02d}-{dt.day:02d}.json'
    fpath = DATA_DIR / fname
    if not fpath.exists():
        print(f'  [warn] missing {fpath}, skipping')
        continue
    _, _, anchors = load_day_signed(
        str(fpath), levels=1, price_level=2.0,
        time_diff_ms=TIME_DIFF_MS, depth=20,
    )
    print(f'  {fname}: {len(anchors):,} snapshots')
    all_anchors.append(anchors)

# Per-day diff so day boundaries don't mix
delta_mid = np.concatenate([np.abs(np.diff(a)) for a in all_anchors])

print(f'\nTotal Δmid samples: {len(delta_mid):,}')
print(f'  median={np.median(delta_mid):.2f}  '
      f'p95={np.percentile(delta_mid, 95):.2f}  '
      f'p99={np.percentile(delta_mid, 99):.2f}  '
      f'max={delta_mid.max():.2f}  USDT')

In [ ]:
OUT_PATH = REPO_ROOT / 'figures' / 'appendix' / 'fig_level_coverage.pdf'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Minimum d to hit each coverage target as δ varies
SWEEP_DELTAS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 10.0])
TARGETS = [
    (0.95, '#BBBBBB', '95%'),
    (0.98, '#CC0000', '98%'),
]

with plt.rc_context({
    'font.family':       'serif',
    'font.size':         9,
    'axes.labelsize':    9,
    'axes.titlesize':    9,
    'legend.fontsize':   7.5,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    ':',
    'figure.dpi':        120,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
}):
    fig, ax = plt.subplots(figsize=(4.0, 3.0))

    d_min_max = 0
    for q, color, lbl in TARGETS:
        width_q = float(np.quantile(delta_mid, q))
        d_min   = np.ceil(width_q / SWEEP_DELTAS).astype(int)
        d_min_max = max(d_min_max, int(d_min.max()))
        lw      = 1.8 if q == 0.98 else 1.1
        ax.plot(SWEEP_DELTAS, d_min, color=color, lw=lw, marker='o',
                markersize=3.5, label=lbl, zorder=3)
        print(f'  target={q*100:.0f}%: width={width_q:.2f} USDT')

    # Highlight chosen (δ=2, d=5)
    ax.axhline(5, color='#0173B2', lw=0.8, ls=':', zorder=1, alpha=0.6)
    ax.axvline(2, color='#0173B2', lw=0.8, ls=':', zorder=1, alpha=0.6)
    ax.scatter([2], [5], s=55, facecolor='#0173B2', edgecolor='white',
               lw=1.0, zorder=5)
    ax.annotate('chosen\n(d=5, δ=2)',
                xy=(2, 5), xytext=(3.0, 8.5),
                color='#0173B2', fontsize=7.5, fontweight='bold',
                ha='left', va='center',
                arrowprops=dict(arrowstyle='-', color='#0173B2', lw=0.8))

    ax.set_xlim(0, 10.5)
    ax.set_ylim(0, d_min_max * 1.08)
    ax.set_xlabel(r'Tick spacing $\delta$ (USDT)', labelpad=4)
    ax.set_ylabel(r'Minimum levels per side $d^\ast$', labelpad=4)
    ax.legend(title='coverage', loc='upper right', frameon=False,
              handlelength=1.5, borderaxespad=0.3)

    fig.tight_layout()
    fig.savefig(OUT_PATH)
    plt.show()

print(f'Saved → {OUT_PATH}')
